# Case Status Transform

This notebook deduplicates long-format case activity rows, reshapes them into a wide case-level output, then computes case-level derived fields and post-wide subcontractor rules from the wide report.

It does the following:
1. Drops duplicate `(Batch Set Name, Batch Name)` pairs, keeping the earliest `Created On` row.
2. Optionally excludes `Batch Name` values found in one or more omit-list CSV files.
3. Builds a wide output with one row per `Batch Name`.
4. Computes `Delivered_to_VA`, `Date_Delivered_to_VA`, `Subcontractor`, and `Latest_Open_BatchSetName` after the wide reshape.
5. Rebuilds eligible `Case Closeout` cases with matching rows from `input_secondary/`.
6. Applies post-wide subcontractor cleanup rules before writing the final CSV.

In [1]:
import csv
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

In [2]:
TERMINAL_BATCH_SET_NAMES = {
    "OMS Closeout",
    "Priority Case Closeout",
    "Case Closeout",
    "FastPass Closeout",
    "FastPass Credentialing",
    "HITL Unworkable",
}

SUBCONTRACTORS = ("XBP", "IRM", "TSS")
TIMESTAMP_FORMAT = "%Y-%m-%d %H:%M:%S.%f %z"
WIDE_TIMESTAMP_FORMAT = "%Y-%m-%d %H:%M:%S"
SECONDARY_TIMESTAMP_FORMATS = ("%m/%d/%y %H:%M", "%m/%d/%y")
MAX_BATCH_SETS = 7

INPUT_PATH = Path("input/")
INPUT_SECONDARY_PATH = Path("secondary_input/")
WIDE_OUTPUT_PATH = Path("output/HITL_billing_wide_07_13.csv")
OMIT_CSV_PATHS = ['omit_files/HITL_Activity_Report_20260116.csv', 'omit_files/2026-04-02-XBP-Closed-Unworkable.csv',
                   'omit_files/FCS-HITL-2026-Feb-Unworkable-1.csv', 'omit_files/FCS-HITL-2026-Feb-Unworkable-2.csv',
                    'omit_files/HITL_Activity_Report_20251202.csv', 'omit_files/HITL_Activity_Report_20260102.csv',
                    'omit_files/XBP_VBMS_UnWorkable_202606.csv']

In [3]:
def resolve_input_csv_paths(input_path: Path) -> List[Path]:
    if input_path.is_file():
        return [input_path]
    if input_path.is_dir():
        csv_paths = sorted(path for path in input_path.rglob("*.csv") if path.is_file())
        if not csv_paths:
            raise ValueError(f"No CSV files were found under input folder: {input_path}")
        return csv_paths
    raise ValueError(f"Input path does not exist or is not readable: {input_path}")


def parse_timestamp(value: str) -> Optional[datetime]:
    cleaned = (value or "").strip()
    if not cleaned or cleaned.upper() == "NULL":
        return None

    for timestamp_format in (TIMESTAMP_FORMAT, WIDE_TIMESTAMP_FORMAT, *SECONDARY_TIMESTAMP_FORMATS):
        try:
            parsed = datetime.strptime(cleaned, timestamp_format)
            if timestamp_format == WIDE_TIMESTAMP_FORMAT or timestamp_format in SECONDARY_TIMESTAMP_FORMATS:
                return parsed.replace(tzinfo=datetime.strptime("+0000", "%z").tzinfo)
            return parsed
        except ValueError:
            continue

    raise ValueError(f"Unsupported timestamp format: {value}")


def normalize_timestamp_string(value: str) -> str:
    cleaned = (value or "").strip()
    if not cleaned or cleaned.upper() == "NULL":
        return ""
    parsed = parse_timestamp(cleaned)
    return parsed.strftime(TIMESTAMP_FORMAT) if parsed is not None else ""


def format_wide_timestamp_string(value: str) -> str:
    cleaned = (value or "").strip()
    if not cleaned or cleaned.upper() == "NULL":
        return ""
    parsed = parse_timestamp(cleaned)
    return parsed.strftime(WIDE_TIMESTAMP_FORMAT) if parsed is not None else ""


def load_rows(input_path: Path) -> Tuple[List[Dict[str, str]], Sequence[str]]:
    rows: List[Dict[str, str]] = []
    fieldnames: List[str] = []
    fieldname_set = set()
    required = {"Batch Name", "Batch Set Name", "Created On"}

    for csv_path in resolve_input_csv_paths(input_path):
        with csv_path.open("r", encoding="utf-8-sig", newline="") as handle:
            reader = csv.DictReader(handle)
            current_fieldnames = list(reader.fieldnames or [])
            missing = required.difference(current_fieldnames)
            if missing:
                raise ValueError(f"Input file is missing required columns {sorted(missing)}: {csv_path}")

            for fieldname in current_fieldnames:
                if fieldname not in fieldname_set:
                    fieldnames.append(fieldname)
                    fieldname_set.add(fieldname)

            for row in reader:
                normalized_row = {fieldname: row.get(fieldname, "") for fieldname in fieldnames}
                for fieldname in current_fieldnames:
                    normalized_row[fieldname] = row.get(fieldname, "")
                for timestamp_field in ("Created On", "Assigned On", "Completed On"):
                    if timestamp_field in normalized_row:
                        normalized_row[timestamp_field] = normalize_timestamp_string(
                            normalized_row.get(timestamp_field, "")
                        )
                rows.append(normalized_row)

    return rows, fieldnames


def write_rows(csv_path: Path, rows: List[Dict[str, str]], fieldnames: Sequence[str]) -> None:
    with csv_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def load_omitted_batch_names(csv_paths: Sequence[Path]) -> set[str]:
    omitted_batch_names: set[str] = set()
    for csv_path in csv_paths:
        with Path(csv_path).open("r", encoding="utf-8-sig", newline="") as handle:
            reader = csv.DictReader(handle)
            fieldnames = set(reader.fieldnames or [])
            if "Batch Name" not in fieldnames:
                raise ValueError(f"Omit file is missing required column 'Batch Name': {csv_path}")
            for row in reader:
                batch_name = (row.get("Batch Name") or "").strip()
                if batch_name:
                    omitted_batch_names.add(batch_name)
    return omitted_batch_names


def parse_created_on(value: str) -> Optional[datetime]:
    return parse_timestamp(value)


def parse_completed_on(value: str) -> Optional[datetime]:
    return parse_timestamp(value)


def row_sort_key(row: Dict[str, str]) -> Tuple[bool, Optional[datetime], str]:
    created_ts = parse_created_on(row.get("Created On", ""))
    batch_set_name = row.get("Batch Set Name", "")
    return (created_ts is None, created_ts, batch_set_name)


def normalize_date_delivered_to_va(value: str) -> str:
    cleaned = (value or "").strip()
    if not cleaned or cleaned.upper() in {"NULL", "N/A"}:
        return ""
    if len(cleaned) == 10 and cleaned.count("-") == 2:
        return cleaned
    parsed = parse_created_on(cleaned)
    return parsed.date().isoformat() if parsed is not None else ""


def deduplicate_rows(rows: List[Dict[str, str]]) -> List[Dict[str, str]]:
    indexed_rows = []
    for source_order, row in enumerate(rows):
        indexed_rows.append(
            {
                **row,
                "_created_ts": parse_created_on(row.get("Created On", "")),
                "_source_order": source_order,
            }
        )

    indexed_rows.sort(
        key=lambda row: (
            row.get("Batch Set Name", ""),
            row.get("Batch Name", ""),
            row["_created_ts"] is None,
            row["_created_ts"],
            row["_source_order"],
        )
    )

    seen = set()
    deduped = []
    for row in indexed_rows:
        key = (row.get("Batch Set Name", ""), row.get("Batch Name", ""))
        if key in seen:
            continue
        seen.add(key)
        deduped.append({k: v for k, v in row.items() if not k.startswith("_")})
    return deduped


def transform_rows(rows: List[Dict[str, str]]) -> List[Dict[str, str]]:
    return deduplicate_rows(rows)


def omit_rows_by_batch_name(rows: List[Dict[str, str]], omitted_batch_names: set[str]) -> List[Dict[str, str]]:
    if not omitted_batch_names:
        return rows
    return [row for row in rows if row.get("Batch Name", "") not in omitted_batch_names]


def unique_batch_names(rows: List[Dict[str, str]]) -> set[str]:
    return {row.get("Batch Name", "") for row in rows if row.get("Batch Name", "")}


def canonical_case_row(row: Dict[str, str]) -> Dict[str, str]:
    return {
        "Batch Name": row.get("Batch Name", ""),
        "Batch Set Name": row.get("Batch Set Name", ""),
        "Created On": normalize_timestamp_string(row.get("Created On", "")),
        "Assigned On": normalize_timestamp_string(row.get("Assigned On", "")),
        "Completed On": normalize_timestamp_string(row.get("Completed On", "")),
    }


def latest_open_batch_set_name(rows: List[Dict[str, str]]) -> str:
    open_rows = [row for row in rows if not (row.get("Completed On", "") or "").strip()]
    if not open_rows:
        return ""

    latest_open_row = sorted(
        open_rows,
        key=lambda row: (
            parse_created_on(row.get("Created On", "")) is None,
            parse_created_on(row.get("Created On", "")),
            row.get("Batch Set Name", ""),
        ),
    )[-1]
    return latest_open_row.get("Batch Set Name", "")


def delivered_to_va_value(rows: List[Dict[str, str]]) -> str:
    return "1" if any(row.get("Batch Set Name", "") in TERMINAL_BATCH_SET_NAMES for row in rows) else "0"


def date_delivered_to_va_value(rows: List[Dict[str, str]]) -> str:
    terminal_rows = [
        row for row in rows if row.get("Batch Set Name", "") in TERMINAL_BATCH_SET_NAMES
    ]
    if not terminal_rows:
        return ""
    earliest_terminal_row = sorted(terminal_rows, key=row_sort_key)[0]
    return normalize_date_delivered_to_va(earliest_terminal_row.get("Created On", ""))


def case_closeout_sort_priority(batch_set_name: str) -> int:
    return 1 if (batch_set_name or "").strip() == "Case Closeout" else 0


def terminus_queue_entries(wide_row: Dict[str, str]) -> List[Tuple[str, Optional[datetime], int]]:
    entries: List[Tuple[str, Optional[datetime], int]] = []
    for idx in range(1, MAX_BATCH_SETS + 1):
        batch_set_name = (wide_row.get(f"BS{idx} Name", "") or "").strip()
        if batch_set_name not in TERMINAL_BATCH_SET_NAMES:
            continue
        created_on = (wide_row.get(f"BS{idx} Created On", "") or "").strip()
        entries.append((batch_set_name, parse_created_on(created_on), idx))
    return entries


def build_terminus_queue_summary_columns(wide_row: Dict[str, str]) -> Dict[str, str]:
    delivered_to_va = wide_row.get("Delivered_to_VA", "")
    entries = terminus_queue_entries(wide_row)
    distinct_queue_names = sorted({batch_set_name for batch_set_name, _, _ in entries})

    if delivered_to_va == "0":
        return {
            "Multiple_Terminus_Queue_Flag": "",
            "Count_Terminus_Queue": "0",
            "Terminus_Queue_Pairs": "",
            "Earliest_Terminus_Queue": "N/A",
            "Latest_Terminus_Queue": "N/A",
        }

    multiple_terminus_queues = "1" if len(distinct_queue_names) > 1 else "0"
    terminus_queue_pairs = ", ".join(distinct_queue_names)
    sorted_entries = sorted(
        entries,
        key=lambda item: (
            item[1] is None,
            item[1],
            item[0],
            item[2],
        ),
    )
    created_on_values = {created_on for _, created_on, _ in entries}
    if len(entries) > 1 and len(created_on_values) == 1:
        earliest_terminus_queue = "SAME CREATED DATE"
        latest_terminus_queue = "SAME CREATED DATE"
    else:
        earliest_terminus_queue = sorted_entries[0][0] if sorted_entries else ""
        latest_terminus_queue = sorted_entries[-1][0] if sorted_entries else ""

    return {
        "Multiple_Terminus_Queue_Flag": multiple_terminus_queues,
        "Count_Terminus_Queue": str(len(entries)),
        "Terminus_Queue_Pairs": terminus_queue_pairs,
        "Earliest_Terminus_Queue": earliest_terminus_queue,
        "Latest_Terminus_Queue": latest_terminus_queue,
    }


def add_post_wide_terminus_queue_columns(wide_rows: List[Dict[str, str]]) -> List[Dict[str, str]]:
    updated_rows = []
    for wide_row in wide_rows:
        updated_row = dict(wide_row)
        updated_row.update(build_terminus_queue_summary_columns(updated_row))
        updated_rows.append(updated_row)
    return updated_rows


def wide_output_columns() -> List[str]:
    columns = [
        "Batch Name",
        "Delivered_to_VA",
        "Date_Delivered_to_VA",
        "Subcontractor",
        "Latest_Open_BatchSetName",
        "Multiple_Terminus_Queue_Flag",
        "Count_Terminus_Queue",
        "Terminus_Queue_Pairs",
        "Earliest_Terminus_Queue",
        "Latest_Terminus_Queue",
    ]
    for idx in range(1, MAX_BATCH_SETS + 1):
        columns.extend(
            [
                f"BS{idx} Name",
                f"BS{idx} Created On",
                f"BS{idx} Completed On",
            ]
        )
    return columns


def transform_long_to_wide(rows: List[Dict[str, str]]) -> List[Dict[str, str]]:
    ranked_rows = []
    for row in rows:
        ranked_rows.append(
            {
                **row,
                "_created_ts": parse_created_on(row.get("Created On", "")),
                "_completed_ts": parse_completed_on(row.get("Completed On", "")),
            }
        )

    ranked_rows.sort(
        key=lambda row: (
            row.get("Batch Name", ""),
            row["_created_ts"] is None,
            row["_created_ts"],
            case_closeout_sort_priority(row.get("Batch Set Name", "")),
            row["_completed_ts"] is None,
            row["_completed_ts"],
            row.get("Batch Set Name", ""),
        )
    )

    grouped_rows: Dict[str, List[Dict[str, str]]] = {}
    for row in ranked_rows:
        grouped_rows.setdefault(row["Batch Name"], []).append(row)

    wide_rows = []
    for batch_name in sorted(grouped_rows):
        group = grouped_rows[batch_name][:MAX_BATCH_SETS]
        wide_row = {column: "" for column in wide_output_columns()}
        wide_row["Batch Name"] = batch_name
        wide_row["Delivered_to_VA"] = delivered_to_va_value(group)
        wide_row["Date_Delivered_to_VA"] = date_delivered_to_va_value(group)
        wide_row["Subcontractor"] = "N/A"
        wide_row["Latest_Open_BatchSetName"] = latest_open_batch_set_name(group)

        for idx, row in enumerate(group, start=1):
            wide_row[f"BS{idx} Name"] = row.get("Batch Set Name", "")
            wide_row[f"BS{idx} Created On"] = format_wide_timestamp_string(row.get("Created On", ""))
            wide_row[f"BS{idx} Completed On"] = format_wide_timestamp_string(row.get("Completed On", ""))

        wide_rows.append(wide_row)

    return wide_rows


def case_closeout_batch_names_needing_secondary_lookup(
    wide_rows: List[Dict[str, str]],
) -> set[str]:
    batch_names: set[str] = set()
    for wide_row in wide_rows:
        if (
            wide_row.get("Delivered_to_VA") == "1"
            and wide_row.get("BS1 Name") == "Case Closeout"
        ):
            batch_name = wide_row.get("Batch Name", "")
            if batch_name:
                batch_names.add(batch_name)
    return batch_names


def rebuild_case_closeout_cases_with_secondary(
    primary_rows: List[Dict[str, str]],
    transformed_rows: List[Dict[str, str]],
    wide_rows: List[Dict[str, str]],
    secondary_input_path: Path,
) -> Tuple[List[Dict[str, str]], set[str], int]:
    candidate_batch_names = case_closeout_batch_names_needing_secondary_lookup(wide_rows)
    if not candidate_batch_names:
        return list(transformed_rows), set(), 0

    secondary_rows, _ = load_rows(secondary_input_path)
    matched_secondary_rows = [
        row for row in secondary_rows if row.get("Batch Name", "") in candidate_batch_names
    ]
    matched_batch_names = {
        row.get("Batch Name", "") for row in matched_secondary_rows if row.get("Batch Name", "")
    }

    if not matched_secondary_rows:
        return list(transformed_rows), set(), 0

    rows_by_batch_name: Dict[str, List[Dict[str, str]]] = {}
    for row in primary_rows:
        rows_by_batch_name.setdefault(row.get("Batch Name", ""), []).append(canonical_case_row(row))

    secondary_rows_by_batch_name: Dict[str, List[Dict[str, str]]] = {}
    for row in matched_secondary_rows:
        secondary_rows_by_batch_name.setdefault(row.get("Batch Name", ""), []).append(canonical_case_row(row))

    updated_rows = [
        dict(row) for row in transformed_rows if row.get("Batch Name", "") not in matched_batch_names
    ]
    secondary_rows_added = 0

    for batch_name in sorted(matched_batch_names):
        original_case_rows = [
            dict(row) for row in transformed_rows if row.get("Batch Name", "") == batch_name
        ]
        combined_case_rows = rows_by_batch_name.get(batch_name, []) + secondary_rows_by_batch_name.get(batch_name, [])
        recomputed_case_rows = transform_rows(combined_case_rows)
        if len(recomputed_case_rows) < len(original_case_rows):
            updated_rows.extend(original_case_rows)
            continue
        updated_rows.extend(recomputed_case_rows)
        secondary_rows_added += len(recomputed_case_rows) - len(original_case_rows)

    return updated_rows, matched_batch_names, secondary_rows_added


def subcontractor_from_batch_set_name(batch_set_name: str) -> str:
    normalized_name = (batch_set_name or "").upper()
    for subcontractor_name in SUBCONTRACTORS:
        if subcontractor_name in normalized_name:
            return subcontractor_name
    return ""


def subcontractor_from_wide_slots(wide_row: Dict[str, str]) -> str:
    bs1_created_on = (wide_row.get("BS1 Created On", "") or "").strip()
    bs2_created_on = (wide_row.get("BS2 Created On", "") or "").strip()
    bs1_completed_on = (wide_row.get("BS1 Completed On", "") or "").strip()
    bs2_completed_on = (wide_row.get("BS2 Completed On", "") or "").strip()

    fieldnames = ["BS1 Name", "BS2 Name"]
    if bs1_created_on and bs1_created_on == bs2_created_on and bool(bs1_completed_on) != bool(bs2_completed_on):
        fieldnames = ["BS1 Name", "BS2 Name"] if bs1_completed_on else ["BS2 Name", "BS1 Name"]

    for fieldname in fieldnames:
        batch_set_name = wide_row.get(fieldname, "")
        if "UNWORKABLE" in (batch_set_name or "").upper():
            return "XBP"
        resolved_subcontractor = subcontractor_from_batch_set_name(batch_set_name)
        if resolved_subcontractor:
            return resolved_subcontractor
    return "N/A"


def apply_post_wide_subcontractor_rules(
    wide_rows: List[Dict[str, str]],
    transformed_rows: List[Dict[str, str]],
    va_batch_names: set[str],
) -> Tuple[List[Dict[str, str]], List[Dict[str, str]]]:
    adjusted_wide_rows = []
    subcontractor_overrides: Dict[str, str] = {}

    for wide_row in wide_rows:
        adjusted_wide_row = dict(wide_row)
        batch_name = adjusted_wide_row.get("Batch Name", "")
        delivered_to_va = adjusted_wide_row.get("Delivered_to_VA")
        current_subcontractor = adjusted_wide_row.get("Subcontractor", "")
        bs1_name = adjusted_wide_row.get("BS1 Name", "")
        wide_slot_subcontractor = subcontractor_from_wide_slots(adjusted_wide_row)

        if bs1_name in {"Priority FOIA", "Priority PA", "FOIA"}:
            adjusted_wide_row["Subcontractor"] = "VA"
            subcontractor_overrides[batch_name] = "VA"
            adjusted_wide_rows.append(adjusted_wide_row)
            continue

        if batch_name in va_batch_names:
            adjusted_wide_row["Subcontractor"] = "VA"
            subcontractor_overrides[batch_name] = "VA"
            current_subcontractor = "VA"

        if current_subcontractor == "VA" and bs1_name == "Case Closeout":
            adjusted_wide_row["Subcontractor"] = "XBP"
            subcontractor_overrides[batch_name] = "XBP"
            current_subcontractor = "XBP"

        if wide_slot_subcontractor != "N/A":
            adjusted_wide_row["Subcontractor"] = wide_slot_subcontractor
            subcontractor_overrides[batch_name] = wide_slot_subcontractor
            current_subcontractor = wide_slot_subcontractor

        if delivered_to_va == "0":
            adjusted_wide_row["Subcontractor"] = wide_slot_subcontractor
            subcontractor_overrides[batch_name] = wide_slot_subcontractor
            adjusted_wide_rows.append(adjusted_wide_row)
            continue

        if delivered_to_va == "1" and current_subcontractor == "N/A":
            if bs1_name == "HITL Unworkable":
                adjusted_wide_row["Subcontractor"] = "XBP"
                subcontractor_overrides[batch_name] = "XBP"
            elif bs1_name in {"FastPass Closeout", "FastPass Credentialing"}:
                bs2_name = adjusted_wide_row.get("BS2 Name", "")
                resolved_subcontractor = subcontractor_from_batch_set_name(bs2_name)
                if resolved_subcontractor:
                    adjusted_wide_row["Subcontractor"] = resolved_subcontractor
                    subcontractor_overrides[batch_name] = resolved_subcontractor

        adjusted_wide_rows.append(adjusted_wide_row)

    adjusted_transformed_rows = []
    for row in transformed_rows:
        batch_name = row.get("Batch Name", "")
        adjusted_row = dict(row)
        if batch_name in subcontractor_overrides:
            adjusted_row["Subcontractor"] = subcontractor_overrides[batch_name]
        adjusted_transformed_rows.append(adjusted_row)

    return adjusted_wide_rows, adjusted_transformed_rows


In [4]:
rows, _ = load_rows(INPUT_PATH)
omitted_batch_names = load_omitted_batch_names([Path(path) for path in OMIT_CSV_PATHS])
input_batch_names = unique_batch_names(rows)
omitted_matches = sorted(input_batch_names.intersection(omitted_batch_names))
primary_rows = omit_rows_by_batch_name(rows, omitted_batch_names)
deduped_rows = deduplicate_rows(primary_rows)
transformed_rows = transform_rows(primary_rows)
primary_rows_after_omit = len(transformed_rows)
primary_unique_batch_names_after_omit = len(unique_batch_names(primary_rows))
wide_rows = transform_long_to_wide(transformed_rows)
candidate_batch_names = case_closeout_batch_names_needing_secondary_lookup(wide_rows)
transformed_rows, va_batch_names, secondary_rows_added = rebuild_case_closeout_cases_with_secondary(
    primary_rows,
    transformed_rows,
    wide_rows,
    INPUT_SECONDARY_PATH,
)
deduped_rows = deduplicate_rows(transformed_rows)
rows_after_recompute_omit = len(transformed_rows)
wide_rows = transform_long_to_wide(transformed_rows)
wide_rows, transformed_rows = apply_post_wide_subcontractor_rules(
    wide_rows,
    transformed_rows,
    va_batch_names,
)
wide_rows = add_post_wide_terminus_queue_columns(wide_rows)
wide_fieldnames = wide_output_columns()

print(f"Input rows: {len(rows)}")
print(f"Input unique batch names: {len(input_batch_names)}")
print(f"Omit list batch names: {len(omitted_batch_names)}")
print(f"Input batch names matched by omit lists: {len(omitted_matches)}")
print(f"Sample omitted batch names: {omitted_matches[:10]}")
print(f"Rows after dedupe: {len(deduped_rows)}")
print(f"Primary rows after omit: {primary_rows_after_omit}")
print(f"Primary unique batch names after omit: {primary_unique_batch_names_after_omit}")
print(f"Case Closeout candidates for secondary lookup: {len(candidate_batch_names)}")
print(f"Matched VA batch names: {len(va_batch_names)}")
print(f"Secondary rows added: {secondary_rows_added}")
print(f"Rows after recompute omit: {rows_after_recompute_omit}")
print(f"Rows after post-wide cleanup: {len(transformed_rows)}")
print(f"Wide rows: {len(wide_rows)}")
print(f"Wide output path: {WIDE_OUTPUT_PATH.resolve()}")

Input rows: 2775522
Input unique batch names: 93458
Omit list batch names: 5997
Input batch names matched by omit lists: 3378
Sample omitted batch names: ['007782502-00217_001', '012565471-02947_001', '04206406-00580_001', '042821465-00418_001', '05146780-00182_001', '05575877-04023_001', '061860083-03357_001', '086660625-02142_001', '087565854-00853_001', '096706807-00209_001']
Rows after dedupe: 339692
Primary rows after omit: 339469
Primary unique batch names after omit: 90080
Case Closeout candidates for secondary lookup: 241
Matched VA batch names: 241
Secondary rows added: 223
Rows after recompute omit: 339692
Rows after post-wide cleanup: 339692
Wide rows: 90080
Wide output path: /Users/Venus.Singh/Downloads/HITL_billing/output/HITL_billing_wide_07_13.csv


In [5]:
transformed_rows[:5]

[{'Batch Group': '1',
  'Level #': '5',
  'Batch Set Name': 'Case Closeout',
  'Batch Name': '(empty)-00311_001',
  '# Docs': '216',
  '# Pages': '3609',
  'Created On': '2026-05-29 16:43:42.050000 -0400',
  'Assigned On': '2026-05-31 07:15:48.713000 -0400',
  'Completed On': '2026-05-31 10:24:07.107000 -0400',
  '# Redactions': '64',
  '# Redacted Pages': '49',
  'Assigned By': 'Auto assigned',
  'Assigned To (UserID)': '603',
  'Assigned To': 'Charles.Herrick@va.gov',
  'Completed By': 'Charles.Herrick@va.gov',
  'Batch Status': 'Complete',
  '# Docs Done': '215',
  '# Docs ToDo': '0',
  '# Docs Not Needed': '1',
  '# Docs Commingled': '0',
  '# Docs Requires Deletion': '0',
  '# Docs Corretions Required': '0',
  '# Docs QA Pass': '0',
  '# Docs Techinical Issues': '0',
  '# Docs Corrupt': '0',
  'Batch_Set_ID': '18',
  'Subcontractor': 'XBP'},
 {'Batch Group': '2',
  'Level #': '5',
  'Batch Set Name': 'Case Closeout',
  'Batch Name': '001334405-00179_001',
  '# Docs': '63',
  '# Pa

In [6]:
wide_rows[:5]

[{'Batch Name': '(empty)-00311_001',
  'Delivered_to_VA': '1',
  'Date_Delivered_to_VA': '2026-05-29',
  'Subcontractor': 'XBP',
  'Latest_Open_BatchSetName': '',
  'BS1 Name': 'XBP VBMS Review',
  'BS1 Created On': '2026-05-22 21:51:26',
  'BS1 Completed On': '2026-05-26 14:39:00',
  'BS2 Name': 'XBP FOIA-PA Redactions',
  'BS2 Created On': '2026-05-26 14:39:00',
  'BS2 Completed On': '2026-05-26 21:28:33',
  'BS3 Name': 'XBP QA',
  'BS3 Created On': '2026-05-26 21:28:33',
  'BS3 Completed On': '2026-05-29 00:57:54',
  'BS4 Name': 'GDIT XBP Production Close-out',
  'BS4 Created On': '2026-05-29 00:57:54',
  'BS4 Completed On': '2026-05-29 16:43:42',
  'BS5 Name': 'Case Closeout',
  'BS5 Created On': '2026-05-29 16:43:42',
  'BS5 Completed On': '2026-05-31 10:24:07',
  'BS6 Name': '',
  'BS6 Created On': '',
  'BS6 Completed On': '',
  'BS7 Name': '',
  'BS7 Created On': '',
  'BS7 Completed On': ''},
 {'Batch Name': '001334405-00179_001',
  'Delivered_to_VA': '1',
  'Date_Delivered_to

In [7]:
write_rows(WIDE_OUTPUT_PATH, wide_rows, wide_fieldnames)
print(f"Wrote wide CSV to: {WIDE_OUTPUT_PATH.resolve()}")

Wrote wide CSV to: /Users/Venus.Singh/Downloads/HITL_billing/output/HITL_billing_wide_07_13.csv
